<a href="https://colab.research.google.com/github/ialejandrozalles/GeneradorCodigoQR/blob/main/GeneradorCodigoQR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install qrcode[pil]
!pip install pillow
!pip install ipywidgets
!pip install numpy
!pip install matplotlib
!pip install opencv-python

In [ ]:
import qrcode
from PIL import Image, ImageDraw, ImageFont
import numpy as np
import matplotlib.pyplot as plt
import io
import base64
from IPython.display import HTML, display
import ipywidgets as widgets
from google.colab import files
import cv2

In [ ]:

class GeneradorQR:
    """
    Clase principal para generar códigos QR personalizados.
    """

    def __init__(self):
        """Inicializa el generador QR con configuración predeterminada."""
        self.colorFondo = "#FFFFFF"
        self.colorQR = "#000000"
        self.tamano = 10
        self.margen = 4
        self.tipoCorreccion = qrcode.constants.ERROR_CORRECT_H
        self.logo = None
        self.logoSize = 0.2

    def configurarColores(self, colorQR="#000000", colorFondo="#FFFFFF"):
        """Configura los colores del código QR."""
        self.colorFondo = colorFondo
        self.colorQR = colorQR
        return self

    def configurarTamano(self, tamano=10, margen=4):
        """Configura el tamaño y márgenes del código QR."""
        self.tamano = tamano
        self.margen = margen
        return self

    def configurarCorreccion(self, nivel="H"):
        """
        Configura el nivel de corrección de errores.
        Niveles: L (7%), M (15%), Q (25%), H (30%)
        """
        niveles = {
            "L": qrcode.constants.ERROR_CORRECT_L,
            "M": qrcode.constants.ERROR_CORRECT_M,
            "Q": qrcode.constants.ERROR_CORRECT_Q,
            "H": qrcode.constants.ERROR_CORRECT_H
        }
        self.tipoCorreccion = niveles.get(nivel, qrcode.constants.ERROR_CORRECT_H)
        return self

    def agregarLogo(self, ruta_logo, tamanoRelativo=0.2):
        """Agrega un logo al centro del código QR."""
        self.logo = ruta_logo
        self.logoSize = tamanoRelativo
        return self

    def generarQR(self, datos):
        """Genera un código QR basado en la configuración actual."""
        qr = qrcode.QRCode(
            version=1,
            error_correction=self.tipoCorreccion,
            box_size=self.tamano,
            border=self.margen
        )

        qr.add_data(datos)
        qr.make(fit=True)

        # Crear imagen QR con colores personalizados
        imagen_qr = qr.make_image(fill_color=self.colorQR, back_color=self.colorFondo).convert('RGBA')

        # Agregar logo si está configurado
        if self.logo is not None:
            try:
                logo = Image.open(self.logo).convert('RGBA')

                # Calcular tamaño del logo basado en el tamaño del QR
                tamano_qr = imagen_qr.size[0]
                tamano_logo = int(tamano_qr * self.logoSize)

                # Redimensionar logo
                logo = logo.resize((tamano_logo, tamano_logo))

                # Calcular posición para centrar el logo
                posicion = ((tamano_qr - tamano_logo) // 2, (tamano_qr - tamano_logo) // 2)

                # Crear una imagen nueva para combinar QR y logo
                imagen_combinada = Image.new('RGBA', imagen_qr.size, (0, 0, 0, 0))
                imagen_combinada.paste(imagen_qr, (0, 0))
                imagen_combinada.paste(logo, posicion, logo)

                return imagen_combinada
            except Exception as e:
                print(f"Error al agregar logo: {e}")
                return imagen_qr

        return imagen_qr

    def guardarQR(self, imagen, nombre_archivo="codigo_qr.png"):
        """Guarda el código QR generado como archivo."""
        if isinstance(imagen, np.ndarray):
            cv2.imwrite(nombre_archivo, imagen)
        else:
            imagen.save(nombre_archivo)
        return nombre_archivo

    def mostrarQR(self, imagen):
        """Muestra el código QR en la salida de Colab."""
        plt.figure(figsize=(8, 8))
        plt.imshow(np.array(imagen))
        plt.axis('off')
        plt.show()

    def descargarQR(self, imagen, nombre_archivo="codigo_qr.png"):
        """Guarda y permite descargar el código QR."""
        nombre_archivo = self.guardarQR(imagen, nombre_archivo)
        files.download(nombre_archivo)

In [ ]:
# Interfaz de usuario para Google Colab
def CrearInterfazQR():
    """Crea una interfaz gráfica para generar códigos QR."""

    # Widgets para datos y configuración
    entradaDatos = widgets.Textarea(description='Datos:', placeholder='Ingrese texto o URL para el código QR', rows=3)
    selectorCorreccion = widgets.Dropdown(
        options=[('Bajo (L)', 'L'), ('Medio (M)', 'M'), ('Cuarto (Q)', 'Q'), ('Alto (H)', 'H')],
        value='H',
        description='Corrección:'
    )
    selectorTamano = widgets.IntSlider(min=5, max=20, value=10, description='Tamaño:')
    selectorMargen = widgets.IntSlider(min=0, max=10, value=4, description='Margen:')
    entradaColorQr = widgets.ColorPicker(concise=False, description='Color QR:', value='#000000')
    entradaColorFondo = widgets.ColorPicker(concise=False, description='Color fondo:', value='#FFFFFF')

    # Botón para subir logo
    botonLogo = widgets.FileUpload(description='Subir logo', accept='image/*', multiple=False)

    # Botón para generar QR
    botonGenerar = widgets.Button(description='Generar QR')
    salidaMensaje = widgets.Output()

    # Salida para mostrar el código QR
    salidaQr = widgets.Output()

    # Función para manejar la generación del QR
    def GenerarQRClick(b):
        with salidaQr:
            salidaQr.clear_output()
            with salidaMensaje:
                salidaMensaje.clear_output()

                if not entradaDatos.value:
                    with salidaMensaje:
                        print("Por favor ingrese datos para generar el código QR")
                    return

                try:
                    print("Generando código QR...")

                    # Configurar generador QR
                    generador = GeneradorQR()
                    generador.configurarColores(entradaColorQr.value, entradaColorFondo.value)
                    generador.configurarTamano(selectorTamano.value, selectorMargen.value)
                    generador.configurarCorreccion(selectorCorreccion.value)

                    # Manejar logo si se ha subido
                    if botonLogo.value:
                        nombreArchivo = list(botonLogo.value.keys())[0]
                        contenido = list(botonLogo.value.values())[0]['content']

                        # Guardar logo temporalmente
                        with open(nombreArchivo, 'wb') as f:
                            f.write(contenido)

                        generador.agregarLogo(nombreArchivo)

                    # Generar código QR
                    qrImagen = generador.generarQR(entradaDatos.value)

                    # Mostrar código QR
                    generador.mostrarQR(qrImagen)

                    # Guardar y ofrecer descargar
                    nombreArchivo = "codigo_qr.png"
                    generador.guardarQR(qrImagen, nombreArchivo)

                    # Botón para descargar
                    botonDescargar = widgets.Button(description='Descargar QR')
                    display(botonDescargar)

                    def DescargarQRClick(b):
                        files.download(nombreArchivo)

                    botonDescargar.on_click(DescargarQRClick)

                except Exception as e:
                    print(f"Error al generar el código QR: {e}")

    # Conectar botón con función
    botonGenerar.on_click(GenerarQRClick)

    # Crear diseño de la interfaz
    pestanas = widgets.Tab()

    # Pestaña de configuración básica
    tabBasico = widgets.VBox([entradaDatos, selectorTamano, selectorMargen])

    # Pestaña de configuración avanzada
    tabAvanzado = widgets.VBox([selectorCorreccion, entradaColorQr, entradaColorFondo, botonLogo])

    # Configurar pestañas
    pestanas.children = [tabBasico, tabAvanzado]
    pestanas.set_title(0, 'Básico')
    pestanas.set_title(1, 'Avanzado')

    # Diseño principal
    diseno = widgets.VBox([
        widgets.HTML("<h2 style='text-align:center;'>Generador de Códigos QR Profesional</h2>"),
        pestanas,
        botonGenerar,
        salidaMensaje,
        salidaQr
    ])

    display(diseno)

In [ ]:
# Ejecutar para mostrar la interfaz
def IniciarGeneradorQR():
    """Inicializa la aplicación de generación de códigos QR."""
    # Instalar dependencias necesarias
    import sys
    import subprocess
    import importlib

    # Lista de paquetes requeridos
    paquetes_requeridos = ["qrcode", "pillow", "ipywidgets"]

    # Instalar paquetes faltantes
    for paquete in paquetes_requeridos:
        try:
            importlib.import_module(paquete.lower())
        except ImportError:
            print(f"Instalando {paquete}...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", paquete])

    # Comprobar específicamente qrcode
    try:
        import qrcode
    except ImportError:
        print("Instalando qrcode desde pip...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "qrcode[pil]"])

    print("¡Bienvenido al Generador de Códigos QR Profesional!")
    print("Cargando interfaz...")
    CrearInterfazQR()

In [9]:
# Ejecutar la aplicación
IniciarGeneradorQR()

Instalando pillow...
¡Bienvenido al Generador de Códigos QR Profesional!
Cargando interfaz...
